In [1]:
# ==========================================================
# CELL 1: MOUNT DRIVE + CLONE/PULL REPO
# ==========================================================
from google.colab import drive
import os, sys

drive.mount('/content/drive')

REPO_URL = "https://github.com/bestoism/skripsi-corn-label-noise"
REPO_DIR = "/content/skripsi-corn-label-noise"

if os.path.exists(REPO_DIR):
    print("🔄 Repo sudah ada, menarik update terbaru...")
    !cd {REPO_DIR} && git pull
else:
    print("⬇️  Clone repo baru...")
    !git clone {REPO_URL} {REPO_DIR}

sys.path.append(REPO_DIR)
os.chdir(REPO_DIR)
print(f"\n✅ Setup selesai. Working dir: {os.getcwd()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔄 Repo sudah ada, menarik update terbaru...
Already up to date.

✅ Setup selesai. Working dir: /content/skripsi-corn-label-noise


In [2]:
# ==========================================================
# CELL 2: INSTALL REQUIREMENTS
# ==========================================================
!pip install -q -r requirements.txt
print("✅ Dependencies terpasang.")

✅ Dependencies terpasang.


In [3]:
# ==========================================================
# CELL 2.5: IMPORT UMUM — dipakai di banyak cell berikutnya
# ==========================================================
import os
import pandas as pd
import numpy as np

In [4]:
# ==========================================================
# CELL 2.6 (BARU): STATUS PIPELINE -- CEK SUDAH SAMPAI MANA
# ==========================================================
# Jalankan kapan saja untuk lihat progres tanpa harus scroll ke atas.
# ==========================================================
from src import config
import os

def cek_status():
    config.set_data_version("v2")
    config.set_proxy(3)  # proxy final

    steps = {
        "1. Data mentah (scraping)": config.RAW_DATA_FILE,
        "2. Data v2 preprocessed": config.TRAIN_RAW_FILE,
        "3. Tabel ablasi proxy (pilot study)": config.PROXY_QUALITY_LOG_FILE,
        "4. Cleaned data (proxy final)": config.TRAIN_CLEANED_HARD_FILE,
        "5. Sample validasi manusia": config.HUMAN_VALIDATION_FILE,
        "6. Hasil validasi manusia (terisi)": config.HUMAN_VALIDATION_RESULT_FILE,
        "7. Progress training M1-M6": config.PROGRESS_FILE,
        "8. Hasil 6 skenario": config.FINAL_RESULTS_TABLE_FILE,
        "9. Uji signifikansi": config.SIGNIFICANCE_TEST_FILE,
    }
    print("📋 STATUS PIPELINE (proxy final: finetuned_corn, data v2)\n")
    for label, path in steps.items():
        status = "✅" if os.path.exists(path) else "⬜"
        print(f"{status} {label}")

cek_status()

📁 Data version aktif: v2
📌 Proxy aktif: [3] finetuned_corn — IndoBERT fine-tuned K-Fold, CORN loss (P4) -- DEFAULT/FINAL
   Backbone: indobenchmark/indobert-base-p1 | Data: v2
📁 Data version aktif: v2
📌 Proxy aktif: [3] finetuned_corn — IndoBERT fine-tuned K-Fold, CORN loss (P4) -- DEFAULT/FINAL
   Backbone: indobenchmark/indobert-base-p1 | Data: v2
📋 STATUS PIPELINE (proxy final: finetuned_corn, data v2)

✅ 1. Data mentah (scraping)
✅ 2. Data v2 preprocessed
✅ 3. Tabel ablasi proxy (pilot study)
✅ 4. Cleaned data (proxy final)
⬜ 5. Sample validasi manusia
⬜ 6. Hasil validasi manusia (terisi)
⬜ 7. Progress training M1-M6
⬜ 8. Hasil 6 skenario
⬜ 9. Uji signifikansi


In [5]:
# ==========================================================
# CELL 3: SCRAPING GOOGLE PLAY STORE
# ==========================================================
# ⚠️ JALANKAN SEKALI SAJA SEUMUR PROYEK. RAW_DATA_FILE TIDAK versioned
# (sumber mentah tunggal untuk semua DATA_VERSION) -- jangan dijalankan
# ulang setelah selesai.
# ==========================================================
from src import config

if os.path.exists(config.RAW_DATA_FILE):
    print(f"✅ {config.RAW_DATA_FILE} sudah ada -- scraping dilewati.")
    print("   Hapus file ini manual kalau memang mau scraping ulang dari nol.")
else:
    from scripts.scrape_google_play import main as run_scraping
    run_scraping()

✅ /content/drive/MyDrive/SKRIPSI_CORN/data/raw/all_reviews_master.csv sudah ada -- scraping dilewati.
   Hapus file ini manual kalau memang mau scraping ulang dari nol.


In [6]:
# ==========================================================
# CELL 3.5 (BARU): MIGRASI DATA LAMA -> DATA_VERSION="v1"
# ==========================================================
# File preprocessing lama (sebelum fix kamus slang + dedup konflik) masih
# ada di Drive dengan nama TANPA suffix versi (reviews_clean.csv,
# split_train_raw.csv, split_test.csv). Supaya bisa dipakai sebagai
# pembanding "v1" di ablasi Cell 6.6 nanti, kita salin (bukan pindah)
# ke nama baru yang sesuai skema DATA_VERSION.
#
# JALANKAN SEKALI SAJA. Kalau file lama tidak ada (proyek baru dari nol,
# belum pernah preprocessing sebelumnya), cell ini otomatis dilewati --
# artinya ablasi v1-vs-v2 di Cell 6.6 tidak relevan untukmu, langsung
# lanjut pakai v2 saja.
# ==========================================================
import shutil
from src import config

_OLD_CLEAN = os.path.join(config.DATA_PROCESSED_DIR, "reviews_clean.csv")
_OLD_TRAIN = os.path.join(config.DATA_PROCESSED_DIR, "split_train_raw.csv")
_OLD_TEST  = os.path.join(config.DATA_PROCESSED_DIR, "split_test.csv")

config.set_data_version("v1")
_migrated = 0
for old_path, new_path in [
    (_OLD_CLEAN, config.CLEAN_TEXT_FILE),
    (_OLD_TRAIN, config.TRAIN_RAW_FILE),
    (_OLD_TEST, config.TEST_FILE),
]:
    if os.path.exists(old_path) and not os.path.exists(new_path):
        shutil.copy(old_path, new_path)
        print(f"📋 Disalin: {old_path} -> {new_path}")
        _migrated += 1
    elif os.path.exists(new_path):
        print(f"✅ Sudah ada: {new_path}")
    else:
        print(f"⚠️ Tidak ditemukan (dilewati): {old_path}")

if _migrated == 0 and not os.path.exists(config.TRAIN_RAW_FILE):
    print("\nℹ️  Tidak ada data v1 untuk dimigrasikan -- proyek dimulai dari nol.")
    print("   Ablasi v1-vs-v2 (Cell 6.6) tidak relevan, lanjut langsung ke v2.")

config.set_data_version("v2")  # kembalikan ke default kerja

📁 Data version aktif: v1
⚠️ Tidak ditemukan (dilewati): /content/drive/MyDrive/SKRIPSI_CORN/data/processed/reviews_clean.csv
⚠️ Tidak ditemukan (dilewati): /content/drive/MyDrive/SKRIPSI_CORN/data/processed/split_train_raw.csv
⚠️ Tidak ditemukan (dilewati): /content/drive/MyDrive/SKRIPSI_CORN/data/processed/split_test.csv

ℹ️  Tidak ada data v1 untuk dimigrasikan -- proyek dimulai dari nol.
   Ablasi v1-vs-v2 (Cell 6.6) tidak relevan, lanjut langsung ke v2.
📁 Data version aktif: v2


In [7]:
# ==========================================================
# CELL 3.6 (BARU): MIGRASI VALIDASI MANUSIA YANG SUDAH DIISI -> v1
# ==========================================================
# Kerja manual mengisi human_verdict untuk 50 sample JANGAN sampai hilang
# cuma karena skema penamaan file berubah. Sesuaikan proxy_name di bawah
# kalau validasi manusia lama kamu itu untuk proxy selain P4.
import shutil
from src import config

_OLD_HV_SAMPLE = os.path.join(config.HUMAN_VALIDATION_DIR, "human_validation_sample.csv")
_OLD_HV_RESULT = os.path.join(config.HUMAN_VALIDATION_DIR, "human_validation_result.csv")

config.set_data_version("v1")
config.set_proxy(3)  # ganti kalau validasi manusia lama itu untuk proxy lain

for old_path, new_path in [
    (_OLD_HV_SAMPLE, config.HUMAN_VALIDATION_FILE),
    (_OLD_HV_RESULT, config.HUMAN_VALIDATION_RESULT_FILE),
]:
    if os.path.exists(old_path) and not os.path.exists(new_path):
        shutil.copy(old_path, new_path)
        print(f"📋 Disalin: {old_path} -> {new_path}")
    elif os.path.exists(new_path):
        print(f"✅ Sudah ada: {new_path}")
    else:
        print(f"⚠️ Tidak ditemukan: {old_path}")

config.set_data_version("v2")  # kembalikan ke default kerja

📁 Data version aktif: v1
📌 Proxy aktif: [3] finetuned_corn — IndoBERT fine-tuned K-Fold, CORN loss (P4) -- DEFAULT/FINAL
   Backbone: indobenchmark/indobert-base-p1 | Data: v1
⚠️ Tidak ditemukan: /content/drive/MyDrive/SKRIPSI_CORN/human_validation/human_validation_sample.csv
⚠️ Tidak ditemukan: /content/drive/MyDrive/SKRIPSI_CORN/human_validation/human_validation_result.csv
📁 Data version aktif: v2


In [8]:
# ==========================================================
# CELL 3.7 (BARU): RESET PAKSA FILE v2 -- JALANKAN HANYA SEKALI
# ==========================================================
# ⚠️ Cell ini MENGHAPUS hasil preprocessing v2 yang ada, memaksa Cell 4
# membangunnya ulang dari nol. HANYA jalankan ini kalau kamu SENGAJA mau
# rebuild v2 dari awal (mis. setelah ubah logika preprocessing). JANGAN
# jalankan tanpa sadar di run rutin -- ini TIDAK menyentuh data v1,
# raw data, atau human_validation, jadi aman dari sisi itu.
# ==========================================================
import os
from src import config
config.set_data_version("v2")
for f in [config.CLEAN_TEXT_FILE, config.TRAIN_RAW_FILE, config.TEST_FILE]:
    if os.path.exists(f):
        os.remove(f)
        print(f"🗑️ Dihapus: {f}")

📁 Data version aktif: v2
🗑️ Dihapus: /content/drive/MyDrive/SKRIPSI_CORN/data/processed/reviews_clean__v2.csv
🗑️ Dihapus: /content/drive/MyDrive/SKRIPSI_CORN/data/processed/split_train_raw__v2.csv
🗑️ Dihapus: /content/drive/MyDrive/SKRIPSI_CORN/data/processed/split_test__v2.csv


In [9]:
from src import config
import os

for d in [config.DATA_RAW_DIR, config.DATA_PROCESSED_DIR, config.PROXY_CACHE_DIR,
          config.CLEANED_DIR, config.MODEL_CKPT_ROOT, config.HUMAN_VALIDATION_DIR,
          config.RESULTS_DIR, config.LOGS_DIR]:
    os.makedirs(d, exist_ok=True)
    print(f"✅ {d}")

✅ /content/drive/MyDrive/SKRIPSI_CORN/data/raw
✅ /content/drive/MyDrive/SKRIPSI_CORN/data/processed
✅ /content/drive/MyDrive/SKRIPSI_CORN/proxy_cache
✅ /content/drive/MyDrive/SKRIPSI_CORN/cleaned
✅ /content/drive/MyDrive/SKRIPSI_CORN/models_ckpt
✅ /content/drive/MyDrive/SKRIPSI_CORN/human_validation
✅ /content/drive/MyDrive/SKRIPSI_CORN/results
✅ /content/drive/MyDrive/SKRIPSI_CORN/logs


In [10]:
# ==========================================================
# CELL 4: PREPROCESSING v2 -- fix kamus slang + dedup konflik teks-rating
# ==========================================================
from sklearn.model_selection import train_test_split
from src.preprocess import run_preprocessing
from src import config

config.set_data_version("v2")

if os.path.exists(config.TRAIN_RAW_FILE) and os.path.exists(config.TEST_FILE):
    print(f"✅ Split {config.DATA_VERSION} sudah ada -- preprocessing dilewati.")
    df_train = pd.read_csv(config.TRAIN_RAW_FILE)
    df_test = pd.read_csv(config.TEST_FILE)
    print(f"   Train: {len(df_train)} baris | Test: {len(df_test)} baris")
else:
    print("=" * 60)
    print(f" PREPROCESSING ({config.DATA_VERSION}) ")
    print("=" * 60)
    df_clean = run_preprocessing(config.RAW_DATA_FILE, config.CLEAN_TEXT_FILE)
    df_clean = df_clean.dropna(subset=["cleaned_text", "rating"])

    print("\n" + "=" * 60)
    print(" SPLIT DATA (80% train, 20% test, stratified by rating, seed=42) ")
    print("=" * 60)
    df_train, df_test = train_test_split(
        df_clean, test_size=0.2, random_state=42, stratify=df_clean["rating"]
    )
    df_train.to_csv(config.TRAIN_RAW_FILE, index=False)
    df_test.to_csv(config.TEST_FILE, index=False)

    print(f"✅ Train: {len(df_train)} baris -> {config.TRAIN_RAW_FILE}")
    print(f"✅ Test : {len(df_test)} baris -> {config.TEST_FILE}")

📖 Kamus slang dasar: 15006 entri (Salsabila dkk., 2018)
📁 Data version aktif: v2
 PREPROCESSING (v2) 
📥 Membaca data mentah dari: /content/drive/MyDrive/SKRIPSI_CORN/data/raw/all_reviews_master.csv
📏 Jumlah data awal: 10800 baris
🧹 Cleaning teks (lowercase, URL/tag, emoji, elongasi, slang)...
📊 Cakupan kamus slang: 16099/138985 kata (11.58%)
🔍 Teks identik, rating berbeda: 164 kasus
🗑️  Baris dibuang (rating minoritas dalam grup konflik): 1005
🗑️  Baris dibuang (tie, tidak ada mayoritas jelas): 181
   Total dibuang: 1186
✅ Setelah dibersihkan: 8562 baris (terbuang: 2238)
💾 Disimpan di: /content/drive/MyDrive/SKRIPSI_CORN/data/processed/reviews_clean__v2.csv
💾 Ringkasan preprocessing -> /content/drive/MyDrive/SKRIPSI_CORN/results/preprocessing_summary__v2.csv

 SPLIT DATA (80% train, 20% test, stratified by rating, seed=42) 
✅ Train: 6849 baris -> /content/drive/MyDrive/SKRIPSI_CORN/data/processed/split_train_raw__v2.csv
✅ Test : 1713 baris -> /content/drive/MyDrive/SKRIPSI_CORN/data/pr

In [11]:
# ==========================================================
# CELL 4.5 (BARU): BANDINGKAN UKURAN & CAKUPAN SLANG v1 vs v2
# ==========================================================
from src import config

_summary_v1 = os.path.join(config.RESULTS_DIR, "preprocessing_summary__v1.csv")
_summary_v2 = os.path.join(config.RESULTS_DIR, "preprocessing_summary__v2.csv")

print("📊 Perbandingan preprocessing v1 (lama, ada bug) vs v2 (sudah di-fix):\n")
if os.path.exists(_summary_v1):
    display(pd.read_csv(_summary_v1))
else:
    print("   (ringkasan v1 tidak ada -- kemungkinan proyek dimulai dari nol)")

if os.path.exists(_summary_v2):
    display(pd.read_csv(_summary_v2))
else:
    print("   ⚠️ Ringkasan v2 belum ada -- pastikan Cell 4 sudah dijalankan.")

📊 Perbandingan preprocessing v1 (lama, ada bug) vs v2 (sudah di-fix):

   (ringkasan v1 tidak ada -- kemungkinan proyek dimulai dari nol)


,initial_rows,final_rows,rows_dropped,text_rating_conflicts,slang_total_words,slang_normalized_words,slang_coverage_pct
0,10800,8562,2238,164,138985,16099,11.58


In [12]:
# ==========================================================
# CELL 5: PILOT STUDY -- ABLASI PROXY 0-3 (data v2)
# ==========================================================
import pandas as pd
from src import config
from src.clean import run_confident_learning

config.set_data_version("v2")

PILOT_PROXY_IDS = [0, 1, 2, 3]  # 4 (fusion) & 6 (IndoBERTweet) dijalankan terpisah di cell lain

for pid in PILOT_PROXY_IDS:
    print(f"\n{'='*70}\n PILOT STUDY -- PROXY_ID = {pid} | DATA = {config.DATA_VERSION} \n{'='*70}")
    config.set_proxy(pid)
    try:
        run_confident_learning()
    except Exception as e:
        print(f"⚠️ Proxy {pid} gagal: {e}")
        continue

print("\n✅ Pilot study selesai.")
pilot_table = pd.read_csv(config.PROXY_QUALITY_LOG_FILE)
display(pilot_table[pilot_table["data_version"] == "v2"])

📁 Data version aktif: v2

 PILOT STUDY -- PROXY_ID = 0 | DATA = v2 
📌 Proxy aktif: [0] frozen_cls_lr — CLS embedding beku + LR (P1)
   Backbone: indobenchmark/indobert-base-p1 | Data: v2
 CONFIDENT LEARNING — proxy aktif: [0] frozen_cls_lr | data: v2 
📥 Memuat 6849 baris data train.

🧮 Menghitung OOF pred_probs — proxy [0] frozen_cls_lr
⚡ Memuat cache proxy [frozen_cls_lr] ...

📐 Kualitas proxy [frozen_cls_lr] (data v2):
   Exact Accuracy : 0.4303
   MAE            : 0.9366
   Off-by-1 Acc   : 0.7497
   QWK            : 0.6092

🔎 Analisis Metode Filter Cleanlab:
   'confident_learning': 3209 baris diflag (46.85%)
   'prune_by_noise_rate': 2735 baris diflag (39.93%)

✅ Deteksi selesai (metode utama: confident_learning).
   Hard-prune     : buang 3209 / sisa 3640
   Severity-aware : buang 1351 / sisa 5498

📊 Distribusi rating_diff pada baris noise:
rating_diff
1    1858
2     860
3     386
4     105
Name: count, dtype: int64
📄 Tabel ablasi proxy diperbarui -> /content/drive/MyDrive/SKRIP

,proxy_id,proxy_name,proxy_desc,data_version,accuracy,mae,off_by_one,qwk,pct_flagged_noise
0,0,frozen_cls_lr,CLS embedding beku + LR (P1),v2,0.430282,0.936633,0.749744,0.609230,46.853555
1,1,frozen_meanpool_lr,Mean-pool embedding beku + LR (P2),v2,0.430428,0.928019,0.758651,0.612447,47.233173
2,2,finetuned_ce,"IndoBERT fine-tuned K-Fold, CE loss (P3)",v2,0.445174,0.785078,0.830194,0.669671,52.577019
3,3,finetuned_corn,"IndoBERT fine-tuned K-Fold, CORN loss (P4) -- ...",v2,0.451015,0.771938,0.831070,0.682986,51.700978
4,4,finetuned_corn_fusion,IndoBERT+CORN + fusi sentimen (P5),v2,0.454519,0.767557,0.833552,0.694756,50.620529


In [13]:
# ==========================================================
# CELL 6: PROXY FINAL SEMENTARA (P4, CORN) -- data v2
# ==========================================================
# ⚠️ RESTART RUNTIME dulu sebelum cell ini kalau tadi jalankan Cell 5
# (loop pilot study), supaya config bersih.
# ==========================================================
from src import config

config.set_data_version("v2")
config.set_proxy(3)
print(f"📌 Proxy sementara: [{config.PROXY_ID}] {config.PROXY_NAME} | data {config.DATA_VERSION}")

from src.clean import run_confident_learning
df_noise, proxy_metrics = run_confident_learning()

📁 Data version aktif: v2
📌 Proxy aktif: [3] finetuned_corn — IndoBERT fine-tuned K-Fold, CORN loss (P4) -- DEFAULT/FINAL
   Backbone: indobenchmark/indobert-base-p1 | Data: v2
📌 Proxy sementara: [3] finetuned_corn | data v2
 CONFIDENT LEARNING — proxy aktif: [3] finetuned_corn | data: v2 
📥 Memuat 6849 baris data train.

🧮 Menghitung OOF pred_probs — proxy [3] finetuned_corn
⚡ Memuat cache proxy [finetuned_corn] ...

📐 Kualitas proxy [finetuned_corn] (data v2):
   Exact Accuracy : 0.4510
   MAE            : 0.7719
   Off-by-1 Acc   : 0.8311
   QWK            : 0.6830

🔎 Analisis Metode Filter Cleanlab:
   'confident_learning': 3541 baris diflag (51.70%)
   'prune_by_noise_rate': 3115 baris diflag (45.48%)

✅ Deteksi selesai (metode utama: confident_learning).
   Hard-prune     : buang 3541 / sisa 3308
   Severity-aware : buang 1098 / sisa 5751

📊 Distribusi rating_diff pada baris noise:
rating_diff
1    2443
2     828
3     219
4      51
Name: count, dtype: int64
📄 Tabel ablasi proxy d

In [14]:
# ==========================================================
# CELL 6.5: CEK KELENGKAPAN FILE DI DRIVE
# ==========================================================
from src import config
import os

checks = {
    "Data train (v2)": config.TRAIN_RAW_FILE,
    "Data test (v2)": config.TEST_FILE,
    "Tabel ablasi proxy": config.PROXY_QUALITY_LOG_FILE,
}

all_ok = True
for label, path in checks.items():
    exists = os.path.exists(path)
    status = "✅" if exists else "❌"
    print(f"{status} {label}: {path}")
    if not exists:
        all_ok = False

if all_ok:
    print("\n✅ Semua file ditemukan -- aman lanjut ke Cell 6.6.")
else:
    print("\n⛔ Ada file tidak ditemukan! Cek akun Drive yang dipakai.")

✅ Data train (v2): /content/drive/MyDrive/SKRIPSI_CORN/data/processed/split_train_raw__v2.csv
✅ Data test (v2): /content/drive/MyDrive/SKRIPSI_CORN/data/processed/split_test__v2.csv
✅ Tabel ablasi proxy: /content/drive/MyDrive/SKRIPSI_CORN/results/proxy_ablation_table.csv

✅ Semua file ditemukan -- aman lanjut ke Cell 6.6.


In [15]:
# ==========================================================
# CELL 6.6 (BARU): ABLASI EFEK FIX DATA -- P4 di v1 vs v2
# ==========================================================
# Satu variabel yang berubah: DATA_VERSION. Metode proxy tetap sama (P4).
# Ini mengukur murni efek fix kamus slang + dedup konflik teks-rating,
# terpisah dari eksperimen backbone (Cell 6.7 & 6.8).
#
# CATATAN: karena cache OOF proxy 3 di v1 belum pernah dihitung dengan
# nama file baru (oof_pred_probs__finetuned_corn__v1.npy), cell ini akan
# fine-tune ulang dari nol untuk v1 (bukan dari cache) -- wajar, sekali
# saja, dan hasilnya deterministik (seed=42) sehingga sebanding dengan
# angka lama yang sudah kamu punya di laporan sebelumnya.
# ==========================================================
from src import config
from src.clean import run_confident_learning

if not os.path.exists(os.path.join(config.DATA_PROCESSED_DIR, "split_train_raw__v1.csv")):
    print("ℹ️  Data v1 tidak tersedia (lihat Cell 3.5) -- ablasi ini dilewati.")
else:
    config.set_data_version("v1")
    config.set_proxy(3)
    print(f"\n{'='*70}\n P4 DI DATA v1 (sebelum fix) \n{'='*70}")
    df_noise_v1, metrics_v1 = run_confident_learning()

    config.set_data_version("v2")
    config.set_proxy(3)
    print(f"\n{'='*70}\n P4 DI DATA v2 (sesudah fix) \n{'='*70}")
    df_noise_v2, metrics_v2 = run_confident_learning()

    print("\n" + "=" * 60)
    print(" PERBANDINGAN EFEK FIX DATA (P4, proxy sama) ")
    print("=" * 60)
    print(f"v1 (sebelum fix): Acc={metrics_v1['accuracy']:.4f} | MAE={metrics_v1['mae']:.4f} "
          f"| Off-by-1={metrics_v1['off_by_one']:.4f} | QWK={metrics_v1['qwk']:.4f}")
    print(f"v2 (sesudah fix): Acc={metrics_v2['accuracy']:.4f} | MAE={metrics_v2['mae']:.4f} "
          f"| Off-by-1={metrics_v2['off_by_one']:.4f} | QWK={metrics_v2['qwk']:.4f}")

    # kembalikan ke v2 -- default kerja untuk cell selanjutnya
    config.set_data_version("v2")
    config.set_proxy(3)

ℹ️  Data v1 tidak tersedia (lihat Cell 3.5) -- ablasi ini dilewati.


In [16]:
# ==========================================================
# CELL 6.7: ABLASI TAMBAHAN -- P5 (FUSION SENTIMEN), data v2
# ==========================================================
import traceback
from src import config
from src.clean import run_confident_learning

config.set_data_version("v2")
config.set_proxy(4)
try:
    df_noise_p5, proxy_metrics_p5 = run_confident_learning()
except Exception as e:
    print(f"⚠️ Proxy 4 (fusion) gagal:")
    traceback.print_exc()
finally:
    config.set_proxy(3)
    print(f"\n📌 Proxy dikembalikan ke sementara: [{config.PROXY_ID}] {config.PROXY_NAME}")

📁 Data version aktif: v2
📌 Proxy aktif: [4] finetuned_corn_fusion — IndoBERT+CORN + fusi sentimen (P5)
   Backbone: indobenchmark/indobert-base-p1 | Data: v2
 CONFIDENT LEARNING — proxy aktif: [4] finetuned_corn_fusion | data: v2 
📥 Memuat 6849 baris data train.

🧮 Menghitung OOF pred_probs — proxy [4] finetuned_corn_fusion
⚡ Memuat cache proxy [finetuned_corn_fusion] ...

📐 Kualitas proxy [finetuned_corn_fusion] (data v2):
   Exact Accuracy : 0.4545
   MAE            : 0.7676
   Off-by-1 Acc   : 0.8336
   QWK            : 0.6948

🔎 Analisis Metode Filter Cleanlab:
   'confident_learning': 3467 baris diflag (50.62%)
   'prune_by_noise_rate': 3145 baris diflag (45.92%)

✅ Deteksi selesai (metode utama: confident_learning).
   Hard-prune     : buang 3467 / sisa 3382
   Severity-aware : buang 1070 / sisa 5779

📊 Distribusi rating_diff pada baris noise:
rating_diff
1    2397
2     785
3     226
4      59
Name: count, dtype: int64
📄 Tabel ablasi proxy diperbarui -> /content/drive/MyDrive/SK

In [17]:
# ==========================================================
# CELL 6.8 (BARU): ABLASI BACKBONE -- P6 (IndoBERTweet + CORN), data v2
# ==========================================================
import traceback
from src import config
from src.clean import run_confident_learning

config.set_data_version("v2")
config.set_proxy(6)
print(f"📌 Proxy: [{config.PROXY_ID}] {config.PROXY_NAME} | Backbone: {config.PRETRAINED_MODEL_NAME}")
try:
    df_noise_p6, proxy_metrics_p6 = run_confident_learning()
except Exception as e:
    print(f"⚠️ Proxy 6 (IndoBERTweet) gagal:")
    traceback.print_exc()
finally:
    config.set_proxy(3)
    print(f"\n📌 Proxy dikembalikan ke sementara: [{config.PROXY_ID}] {config.PROXY_NAME}")

📁 Data version aktif: v2
📌 Proxy aktif: [6] finetuned_corn_indobertweet — IndoBERTweet fine-tuned K-Fold, CORN loss (P6)
   Backbone: indolem/indobertweet-base-uncased | Data: v2
📌 Proxy: [6] finetuned_corn_indobertweet | Backbone: indolem/indobertweet-base-uncased
 CONFIDENT LEARNING — proxy aktif: [6] finetuned_corn_indobertweet | data: v2 
📥 Memuat 6849 baris data train.

🧮 Menghitung OOF pred_probs — proxy [6] finetuned_corn_indobertweet
   [Proxy finetuned_corn_indobertweet] Fold 1/5 (train=5479, val=1370)...


config.json:   0%|          | 0.00/1.10k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  445MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  445MB            

...

[transformers] BertModel LOAD REPORT from: indolem/indobertweet-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


      Epoch 1/3 - Loss: 0.5136
      Epoch 2/3 - Loss: 0.4399
      Epoch 3/3 - Loss: 0.3885
   [Proxy finetuned_corn_indobertweet] Fold 5/5 (train=5480, val=1369)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: indolem/indobertweet-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


      Epoch 1/3 - Loss: 0.5119
      Epoch 2/3 - Loss: 0.4427
      Epoch 3/3 - Loss: 0.3897

📐 Kualitas proxy [finetuned_corn_indobertweet] (data v2):
   Exact Accuracy : 0.4593
   MAE            : 0.7579
   Off-by-1 Acc   : 0.8368
   QWK            : 0.6956

🔎 Analisis Metode Filter Cleanlab:
   'confident_learning': 3265 baris diflag (47.67%)
   'prune_by_noise_rate': 2973 baris diflag (43.41%)

✅ Deteksi selesai (metode utama: confident_learning).
   Hard-prune     : buang 3265 / sisa 3584
   Severity-aware : buang 958 / sisa 5891

📊 Distribusi rating_diff pada baris noise:
rating_diff
1    2307
2     719
3     185
4      54
Name: count, dtype: int64
📄 Tabel ablasi proxy diperbarui -> /content/drive/MyDrive/SKRIPSI_CORN/results/proxy_ablation_table.csv

💾 Cleaned (hard)   -> /content/drive/MyDrive/SKRIPSI_CORN/cleaned/train_cleaned_hard__finetuned_corn_indobertweet__v2.csv
💾 Cleaned (severe) -> /content/drive/MyDrive/SKRIPSI_CORN/cleaned/train_cleaned_severe__finetuned_corn_indober

In [18]:
# ==========================================================
# CELL 6.9 (BARU): RINGKASAN SEMUA PROXY DI DATA v2 -- PILIH FINAL DI SINI
# ==========================================================
from src import config

table = pd.read_csv(config.PROXY_QUALITY_LOG_FILE)
table_v2 = table[table["data_version"] == "v2"].sort_values("qwk", ascending=False)
print("📊 Semua proxy yang sudah diuji di data v2, diurutkan dari QWK tertinggi:")
display(table_v2)

print("\n⚠️  TENTUKAN PROXY_ID FINAL secara manual berdasarkan tabel di atas,")
print("   lalu isi di FINAL_PROXY_ID di bawah sebelum lanjut ke Cell 7.")

📊 Semua proxy yang sudah diuji di data v2, diurutkan dari QWK tertinggi:


,proxy_id,proxy_name,proxy_desc,data_version,accuracy,mae,off_by_one,qwk,pct_flagged_noise
5,6,finetuned_corn_indobertweet,"IndoBERTweet fine-tuned K-Fold, CORN loss (P6)",v2,0.459337,0.757921,0.836764,0.695553,47.671193
4,4,finetuned_corn_fusion,IndoBERT+CORN + fusi sentimen (P5),v2,0.454519,0.767557,0.833552,0.694756,50.620529
3,3,finetuned_corn,"IndoBERT fine-tuned K-Fold, CORN loss (P4) -- ...",v2,0.451015,0.771938,0.831070,0.682986,51.700978
2,2,finetuned_ce,"IndoBERT fine-tuned K-Fold, CE loss (P3)",v2,0.445174,0.785078,0.830194,0.669671,52.577019
1,1,frozen_meanpool_lr,Mean-pool embedding beku + LR (P2),v2,0.430428,0.928019,0.758651,0.612447,47.233173
0,0,frozen_cls_lr,CLS embedding beku + LR (P1),v2,0.430282,0.936633,0.749744,0.609230,46.853555



⚠️  TENTUKAN PROXY_ID FINAL secara manual berdasarkan tabel di atas,
   lalu isi di FINAL_PROXY_ID di bawah sebelum lanjut ke Cell 7.


In [19]:
import shutil
from src import config

config.set_proxy(3)
shutil.copy(config.HUMAN_VALIDATION_RESULT_FILE, config.HUMAN_VALIDATION_FILE)
print(f"✅ Dibuat ulang -> {config.HUMAN_VALIDATION_FILE}")

import pandas as pd
df_check = pd.read_csv(config.HUMAN_VALIDATION_FILE)
n_filled = (~df_check["human_verdict"].astype(str).str.strip().isin(["", "nan"])).sum()
print(f"Baris terisi: {n_filled} / {len(df_check)}")

📌 Proxy aktif: [3] finetuned_corn — IndoBERT fine-tuned K-Fold, CORN loss (P4) -- DEFAULT/FINAL
   Backbone: indobenchmark/indobert-base-p1 | Data: v2


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/SKRIPSI_CORN/human_validation/human_validation_result__finetuned_corn__v2.csv'

In [ ]:
import pandas as pd
from src import config

config.set_proxy(3)
df_result = pd.read_csv(config.HUMAN_VALIDATION_RESULT_FILE)
n_filled = (~df_result["human_verdict"].astype(str).str.strip().isin(["", "nan"])).sum()
print(f"Baris terisi: {n_filled} / {len(df_result)}")

In [20]:
# ==========================================================
# CELL 7: KUNCI PROXY FINAL + VALIDASI MANUSIA — WAJIB SEBELUM LANJUT
# ==========================================================
# GANTI angka ini sesuai proxy yang kamu pilih dari tabel Cell 6.9
FINAL_PROXY_ID = 3   # <-- EDIT DI SINI

from src import config
config.set_data_version("v2")
config.set_proxy(FINAL_PROXY_ID)
print(f"🔒 Proxy final terkunci: [{config.PROXY_ID}] {config.PROXY_NAME} | data {config.DATA_VERSION}")

print("\n⚠️  BERHENTI DI SINI SEBELUM LANJUT KE CELL 8 ⚠️")
print(f"1. Buka: {config.HUMAN_VALIDATION_FILE}")
print("2. Isi kolom 'human_verdict' MANUAL untuk SEMUA baris:")
print("   'noise' / 'not_noise' / 'ambiguous'")
print("3. Save file (pastikan tersimpan di Drive, bukan di lokal komputer).")
print("4. Jalankan Cell 7.5 di bawah untuk cek kelengkapan + hitung agreement rate.")

📁 Data version aktif: v2
📌 Proxy aktif: [3] finetuned_corn — IndoBERT fine-tuned K-Fold, CORN loss (P4) -- DEFAULT/FINAL
   Backbone: indobenchmark/indobert-base-p1 | Data: v2
🔒 Proxy final terkunci: [3] finetuned_corn | data v2

⚠️  BERHENTI DI SINI SEBELUM LANJUT KE CELL 8 ⚠️
1. Buka: /content/drive/MyDrive/SKRIPSI_CORN/human_validation/human_validation_sample__finetuned_corn__v2.csv
2. Isi kolom 'human_verdict' MANUAL untuk SEMUA baris:
   'noise' / 'not_noise' / 'ambiguous'
3. Save file (pastikan tersimpan di Drive, bukan di lokal komputer).
4. Jalankan Cell 7.5 di bawah untuk cek kelengkapan + hitung agreement rate.


In [21]:
# ==========================================================
# CELL 7.5: HITUNG AGREEMENT VALIDASI MANUSIA
# ==========================================================
from src.human_validation import compute_agreement

result = compute_agreement()
if result is not None:
    print("\n✅ Validasi manusia lengkap. Siap lanjut ke Cell 8 (training).")
else:
    print("\n⛔ Belum lengkap/ada error -- perbaiki dulu sebelum lanjut.")

   (file dibaca dengan delimiter ',')
⚠️ Masih ada 50 baris yang belum diisi 'human_verdict'.
   Isi manual semuanya, save, lalu jalankan lagi.

⛔ Belum lengkap/ada error -- perbaiki dulu sebelum lanjut.


In [22]:
# ==========================================================
# CELL 7.6 (BARU): DIAGNOSTIK -- CONFUSION MATRIX + F1 PER KELAS PROXY FINAL
# ==========================================================
# Versi resmi di pipeline final (data v2, proxy terkunci) dari diagnostik
# yang sebelumnya cuma ada di notebook terpisah (SKRIPSITEST, data v1 lama).
# Murah -- pakai cache OOF pred_probs yang sudah ada, tidak retrain.
# ==========================================================
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
from src.proxy import get_proxy_pred_probs

df_train_diag = pd.read_csv(config.TRAIN_RAW_FILE)
labels_diag = df_train_diag["rating"].values - 1
texts_diag = df_train_diag["cleaned_text"].tolist()

pred_probs_diag = get_proxy_pred_probs(texts_diag, labels_diag)
preds_diag = np.argmax(pred_probs_diag, axis=1)

print(f"📊 Classification report -- proxy [{config.PROXY_NAME}], data {config.DATA_VERSION}:\n")
report_str = classification_report(
    labels_diag, preds_diag,
    target_names=[f"Bintang {i+1}" for i in range(5)], digits=3,
)
print(report_str)

cm_diag = confusion_matrix(labels_diag, preds_diag)
cm_df_diag = pd.DataFrame(
    cm_diag,
    index=[f"Asli {i+1}" for i in range(5)],
    columns=[f"Pred {i+1}" for i in range(5)],
)
display(cm_df_diag)

# simpan supaya tidak perlu hitung ulang untuk Bab IV
report_path = os.path.join(config.RESULTS_DIR, f"proxy_classification_report__{config.PROXY_NAME}__{config.DATA_VERSION}.txt")
with open(report_path, "w") as f:
    f.write(report_str)
cm_df_diag.to_csv(os.path.join(config.RESULTS_DIR, f"proxy_confusion_matrix__{config.PROXY_NAME}__{config.DATA_VERSION}.csv"))
print(f"\n💾 Tersimpan -> {report_path}")


🧮 Menghitung OOF pred_probs — proxy [3] finetuned_corn
⚡ Memuat cache proxy [finetuned_corn] ...
📊 Classification report -- proxy [finetuned_corn], data v2:

              precision    recall  f1-score   support

   Bintang 1      0.585     0.568     0.576      1841
   Bintang 2      0.305     0.328     0.316      1325
   Bintang 3      0.305     0.298     0.301      1226
   Bintang 4      0.390     0.334     0.360      1307
   Bintang 5      0.612     0.703     0.654      1150

    accuracy                          0.451      6849
   macro avg      0.439     0.446     0.441      6849
weighted avg      0.448     0.451     0.448      6849



,Pred 1,Pred 2,Pred 3,Pred 4,Pred 5
Asli 1,1045,470,223,72,31
Asli 2,389,434,323,139,40
Asli 3,228,326,365,241,66
Asli 4,89,159,246,437,376
Asli 5,36,35,39,232,808



💾 Tersimpan -> /content/drive/MyDrive/SKRIPSI_CORN/results/proxy_classification_report__finetuned_corn__v2.txt


In [24]:
# ==========================================================
# CELL 8: TRAINING 6 SKENARIO x 3 SEED (AUTO-RESUME)
# ==========================================================
import json
import numpy as np
from src.train import run_experiment
from src import config

# Konfirmasi eksplisit -- cegah salah backbone/data tanpa sadar
print(f"🔧 Training M1-M6 akan pakai backbone: {config.PRETRAINED_MODEL_NAME} | "
      f"data: {config.DATA_VERSION} | proxy aktif: {config.PROXY_NAME}")

scenarios = [
    {"name": "M1_Baseline_CE",        "train_path": config.TRAIN_RAW_FILE,           "loss": "ce"},
    {"name": "M2_CleanedHard_CE",     "train_path": config.TRAIN_CLEANED_HARD_FILE,  "loss": "ce"},
    {"name": "M3_CleanedSevere_CE",   "train_path": config.TRAIN_CLEANED_SEVERE_FILE,"loss": "ce"},
    {"name": "M4_Baseline_CORN",      "train_path": config.TRAIN_RAW_FILE,           "loss": "corn"},
    {"name": "M5_CleanedHard_CORN",   "train_path": config.TRAIN_CLEANED_HARD_FILE,  "loss": "corn"},
    {"name": "M6_CleanedSevere_CORN", "train_path": config.TRAIN_CLEANED_SEVERE_FILE,"loss": "corn"},
]

if os.path.exists(config.PROGRESS_FILE):
    with open(config.PROGRESS_FILE) as f:
        saved_progress = json.load(f)
    print("🔄 Progress sebelumnya ditemukan, melanjutkan yang belum selesai...")
else:
    saved_progress = {}
    print("🆕 Memulai training dari awal...")

all_results = []
print(f"\n🔥 {len(scenarios)} SKENARIO x {len(config.SEED_LIST)} SEED 🔥\n")

for scenario in scenarios:
    name = scenario["name"]
    metrics_list = {"mae": [], "rmse": [], "accuracy": [], "off_by_one": [], "qwk": []}
    saved_progress.setdefault(name, {})

    for seed in config.SEED_LIST:
        seed_key = str(seed)
        if seed_key in saved_progress[name]:
            print(f"⏩ {name} | seed {seed} (sudah selesai)")
            metrics = saved_progress[name][seed_key]
        else:
            metrics = run_experiment(name, scenario["train_path"], scenario["loss"], seed)
            saved_progress[name][seed_key] = metrics
            with open(config.PROGRESS_FILE, "w") as f:
                json.dump(saved_progress, f, indent=2)

        for k in metrics_list:
            metrics_list[k].append(metrics[k])

    all_results.append({
        "Model": name,
        "MAE (↓)": f"{np.mean(metrics_list['mae']):.4f} ± {np.std(metrics_list['mae']):.4f}",
        "RMSE (↓)": f"{np.mean(metrics_list['rmse']):.4f} ± {np.std(metrics_list['rmse']):.4f}",
        "Acc (↑)": f"{np.mean(metrics_list['accuracy']):.4f} ± {np.std(metrics_list['accuracy']):.4f}",
        "Off-by-1 (↑)": f"{np.mean(metrics_list['off_by_one']):.4f} ± {np.std(metrics_list['off_by_one']):.4f}",
        "QWK (↑)": f"{np.mean(metrics_list['qwk']):.4f} ± {np.std(metrics_list['qwk']):.4f}",
        "_raw_mae": np.mean(metrics_list["mae"]),
    })

df_final = pd.DataFrame(sorted(all_results, key=lambda x: x["_raw_mae"])).drop(columns=["_raw_mae"])
df_final.to_csv(config.FINAL_RESULTS_TABLE_FILE, index=False)

print("\n" + "=" * 100)
print(f" 🏆 HASIL 6 SKENARIO -- backbone: {config.PRETRAINED_MODEL_NAME} | data: {config.DATA_VERSION} 🏆")
print("=" * 100)
display(df_final)

🔧 Training M1-M6 akan pakai backbone: indobenchmark/indobert-base-p1 | data: v2 | proxy aktif: finetuned_corn
🆕 Memulai training dari awal...

🔥 6 SKENARIO x 3 SEED 🔥


🚀 M1_Baseline_CE | Seed: 42 | Loss: CE


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Epoch 1/8 - Loss: 1.2912 - Val MAE: 0.7679 | QWK: 0.6956 | Off-by-1: 0.8248 | Acc: 0.4657
Epoch 2/8 - Loss: 1.1291 - Val MAE: 0.7358 | QWK: 0.6987 | Off-by-1: 0.8365 | Acc: 0.4774
Epoch 3/8 - Loss: 0.9566 - Val MAE: 0.7737 | QWK: 0.6889 | Off-by-1: 0.8540 | Acc: 0.4336
Epoch 4/8 - Loss: 0.7224 - Val MAE: 0.7372 | QWK: 0.6966 | Off-by-1: 0.8715 | Acc: 0.4423
Epoch 5/8 - Loss: 0.4695 - Val MAE: 0.7971 | QWK: 0.6607 | Off-by-1: 0.8190 | Acc: 0.4657
   ⏹ Early stopping di epoch 5 (Val MAE tidak membaik 3x)
🏆 Test (dari model Val MAE terbaik=0.7358): MAE=0.8021 | RMSE=1.2037 | Acc=0.4495 | Off-by-1=0.8097 | QWK=0.6601

🚀 M1_Baseline_CE | Seed: 123 | Loss: CE


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Epoch 1/8 - Loss: 1.3047 - Val MAE: 0.7650 | QWK: 0.6930 | Off-by-1: 0.8263 | Acc: 0.4672
Epoch 2/8 - Loss: 1.1290 - Val MAE: 0.8102 | QWK: 0.6814 | Off-by-1: 0.8088 | Acc: 0.4423
Epoch 3/8 - Loss: 0.9418 - Val MAE: 0.7912 | QWK: 0.6626 | Off-by-1: 0.8277 | Acc: 0.4350
Epoch 4/8 - Loss: 0.6824 - Val MAE: 0.7839 | QWK: 0.6692 | Off-by-1: 0.8350 | Acc: 0.4175
   ⏹ Early stopping di epoch 4 (Val MAE tidak membaik 3x)
🏆 Test (dari model Val MAE terbaik=0.7650): MAE=0.7846 | RMSE=1.1954 | Acc=0.4618 | Off-by-1=0.8219 | QWK=0.6842

🚀 M1_Baseline_CE | Seed: 2024 | Loss: CE


...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Epoch 1/8 - Loss: 0.4280 - Val MAE: 0.5260 | QWK: 0.8475 | Off-by-1: 0.9497 | Acc: 0.5295
Epoch 2/8 - Loss: 0.3398 - Val MAE: 0.5538 | QWK: 0.8271 | Off-by-1: 0.9531 | Acc: 0.4983
Epoch 3/8 - Loss: 0.2728 - Val MAE: 0.4844 | QWK: 0.8645 | Off-by-1: 0.9479 | Acc: 0.5747
Epoch 4/8 - Loss: 0.2139 - Val MAE: 0.4931 | QWK: 0.8602 | Off-by-1: 0.9653 | Acc: 0.5451
Epoch 5/8 - Loss: 0.1545 - Val MAE: 0.5104 | QWK: 0.8546 | Off-by-1: 0.9531 | Acc: 0.5469
Epoch 6/8 - Loss: 0.1218 - Val MAE: 0.5087 | QWK: 0.8482 | Off-by-1: 0.9635 | Acc: 0.5347
   ⏹ Early stopping di epoch 6 (Val MAE tidak membaik 3x)
🏆 Test (dari model Val MAE terbaik=0.4844): MAE=0.7443 | RMSE=1.1325 | Acc=0.4618 | Off-by-1=0.8447 | QWK=0.7104

🚀 M6_CleanedSevere_CORN | Seed: 2024 | Loss: CORN


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Epoch 1/8 - Loss: 0.4376 - Val MAE: 0.5243 | QWK: 0.8432 | Off-by-1: 0.9288 | Acc: 0.5625
Epoch 2/8 - Loss: 0.3384 - Val MAE: 0.5729 | QWK: 0.8179 | Off-by-1: 0.9253 | Acc: 0.5156
Epoch 3/8 - Loss: 0.2748 - Val MAE: 0.5538 | QWK: 0.8278 | Off-by-1: 0.9340 | Acc: 0.5260
Epoch 4/8 - Loss: 0.2039 - Val MAE: 0.6076 | QWK: 0.7804 | Off-by-1: 0.9236 | Acc: 0.4844
   ⏹ Early stopping di epoch 4 (Val MAE tidak membaik 3x)
🏆 Test (dari model Val MAE terbaik=0.5243): MAE=0.7531 | RMSE=1.1405 | Acc=0.4606 | Off-by-1=0.8365 | QWK=0.7118

 🏆 HASIL 6 SKENARIO -- backbone: indobenchmark/indobert-base-p1 | data: v2 🏆


,Model,MAE (↓),RMSE (↓),Acc (↑),Off-by-1 (↑),QWK (↑)
0,M4_Baseline_CORN,0.7285 ± 0.0140,1.0836 ± 0.0340,0.4485 ± 0.0158,0.8618 ± 0.0184,0.7020 ± 0.0011
1,M2_CleanedHard_CE,0.7429 ± 0.0077,1.1293 ± 0.0074,0.4637 ± 0.0052,0.8432 ± 0.0036,0.6985 ± 0.0074
2,M6_CleanedSevere_CORN,0.7453 ± 0.0060,1.1312 ± 0.0082,0.4639 ± 0.0039,0.8391 ± 0.0040,0.7085 ± 0.0038
3,M5_CleanedHard_CORN,0.7464 ± 0.0051,1.1237 ± 0.0090,0.4598 ± 0.0059,0.8379 ± 0.0029,0.6984 ± 0.0061
4,M3_CleanedSevere_CE,0.7546 ± 0.0193,1.1519 ± 0.0375,0.4639 ± 0.0081,0.8385 ± 0.0153,0.6935 ± 0.0112
5,M1_Baseline_CE,0.7838 ± 0.0153,1.1790 ± 0.0293,0.4516 ± 0.0075,0.8237 ± 0.0122,0.6737 ± 0.0101


In [25]:
# ==========================================================
# CELL 9: UJI SIGNIFIKANSI -- WILCOXON + HOLM-BONFERRONI
# ==========================================================
from src.significance import (
    collect_all_predictions,
    aggregate_errors_across_seeds,
    run_significance_test,
    run_all_effect_sizes,
)
from src import config

print(f"🔧 Menguji hasil untuk backbone: {config.PRETRAINED_MODEL_NAME} | data: {config.DATA_VERSION}")

print("📥 Mengumpulkan prediksi dari 3 seed x 6 skenario...")
true_labels, preds_per_seed = collect_all_predictions(scenarios)

print("\n📊 Mengagregasi absolute error across seed...")
aggregated_errors = aggregate_errors_across_seeds(true_labels, preds_per_seed)

print("\n🔬 Uji Wilcoxon Signed-Rank (3 hipotesis pre-registered) + Holm-Bonferroni...")
sig_results = run_significance_test(aggregated_errors)
display(sig_results)

print("\n📐 Effect size + CI 95% (bootstrap) -- pelengkap p-value...")
effect_sizes = run_all_effect_sizes(aggregated_errors)
display(effect_sizes)

effect_sizes.to_csv(os.path.join(config.RESULTS_DIR, f"effect_sizes__{config.PROXY_NAME}__{config.DATA_VERSION}.csv"), index=False)

🔧 Menguji hasil untuk backbone: indobenchmark/indobert-base-p1 | data: v2
📥 Mengumpulkan prediksi dari 3 seed x 6 skenario...
   Memuat prediksi M1_Baseline_CE | seed 42 ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

   Memuat prediksi M1_Baseline_CE | seed 123 ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

   Memuat prediksi M1_Baseline_CE | seed 2024 ...


...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


📊 Mengagregasi absolute error across seed...

🔬 Uji Wilcoxon Signed-Rank (3 hipotesis pre-registered) + Holm-Bonferroni...

💾 Hasil uji signifikansi -> /content/drive/MyDrive/SKRIPSI_CORN/results/significance_test__finetuned_corn__v2.csv


,hypothesis,model_a,model_b,mean_error_a,mean_error_b,p_value_raw,p_value_corrected,signifikan
0,H1_CORN_vs_CE_raw,M4_Baseline_CORN,M1_Baseline_CE,0.728546,0.783810,0.000001,0.000004,Ya
1,H2_SeverityAware_vs_Base,M6_CleanedSevere_CORN,M4_Baseline_CORN,0.745281,0.728546,0.049302,0.049302,Ya
2,H3_HardPrune_vs_Base,M5_CleanedHard_CORN,M4_Baseline_CORN,0.746449,0.728546,0.021451,0.042903,Ya



📐 Effect size + CI 95% (bootstrap) -- pelengkap p-value...


,model_a,model_b,mean_diff,ci_95_low,ci_95_high
0,M4_Baseline_CORN,M1_Baseline_CE,-0.055264,-0.076095,-0.035021
1,M6_CleanedSevere_CORN,M4_Baseline_CORN,0.016735,-0.002140,0.035805
2,M5_CleanedHard_CORN,M4_Baseline_CORN,0.017902,-0.001557,0.038334


In [26]:
# ==========================================================
# CELL 9.5 (BARU): BREAKDOWN ERROR PER KELAS RATING -- M4 vs M5 vs M6
# ==========================================================
# Menjawab: apakah M5/M6 (hasil cleaning) memburuk KHUSUSNYA di kelas
# rating tengah (bintang 2/3/4), atau merata di semua kelas? Kalau khusus
# di kelas tengah, itu mendukung argumen "cleaning membuang contoh
# sulit/ambigu di kelas tengah", bukan random noise yang tersebar merata.
#
# Pakai ulang true_labels & preds_per_seed dari Cell 9 kalau sudah ada di
# sesi ini -- kalau belum, panggil collect_all_predictions() lagi (cepat,
# cuma forward pass dari checkpoint, TIDAK retrain).
# ==========================================================
import numpy as np
import pandas as pd
from src.significance import collect_all_predictions
from src import config

scenarios = [
    {"name": "M1_Baseline_CE",        "train_path": config.TRAIN_RAW_FILE,           "loss": "ce"},
    {"name": "M2_CleanedHard_CE",     "train_path": config.TRAIN_CLEANED_HARD_FILE,  "loss": "ce"},
    {"name": "M3_CleanedSevere_CE",   "train_path": config.TRAIN_CLEANED_SEVERE_FILE,"loss": "ce"},
    {"name": "M4_Baseline_CORN",      "train_path": config.TRAIN_RAW_FILE,           "loss": "corn"},
    {"name": "M5_CleanedHard_CORN",   "train_path": config.TRAIN_CLEANED_HARD_FILE,  "loss": "corn"},
    {"name": "M6_CleanedSevere_CORN", "train_path": config.TRAIN_CLEANED_SEVERE_FILE,"loss": "corn"},
]

if "true_labels" not in dir() or "preds_per_seed" not in dir():
    print("📥 true_labels/preds_per_seed belum ada di sesi ini, memuat ulang...")
    true_labels, preds_per_seed = collect_all_predictions(scenarios)
else:
    print("⚡ Memakai true_labels/preds_per_seed yang sudah ada dari Cell 9.")

TARGET_SCENARIOS = ["M4_Baseline_CORN", "M5_CleanedHard_CORN", "M6_CleanedSevere_CORN"]

def per_class_breakdown(true_labels, preds_per_seed, scenario_names):
    """MAE & accuracy per kelas rating asli (1-5), dirata-ratakan across 3 seed."""
    rows = []
    for name in scenario_names:
        preds_stack = np.stack(list(preds_per_seed[name].values()))  # (3, n_test)
        for rating in sorted(np.unique(true_labels)):
            mask = true_labels == rating
            mae_per_seed, acc_per_seed = [], []
            for preds in preds_stack:
                errs = np.abs(true_labels[mask] - preds[mask])
                mae_per_seed.append(errs.mean())
                acc_per_seed.append((preds[mask] == true_labels[mask]).mean())
            rows.append({
                "scenario": name, "rating_asli": rating, "n_test": int(mask.sum()),
                "mae_mean": np.mean(mae_per_seed), "mae_std": np.std(mae_per_seed),
                "acc_mean": np.mean(acc_per_seed),
            })
    return pd.DataFrame(rows)

breakdown_df = per_class_breakdown(true_labels, preds_per_seed, TARGET_SCENARIOS)

print("📊 MAE per kelas rating asli, per skenario (rata-rata 3 seed):\n")
pivot_mae = breakdown_df.pivot(index="rating_asli", columns="scenario", values="mae_mean")[TARGET_SCENARIOS]
display(pivot_mae.round(4))

print("\n📊 Selisih MAE vs M4 -- POSITIF = cleaning MEMBUAT LEBIH BURUK di kelas itu:\n")
diff_df = pd.DataFrame({
    "M5_minus_M4": pivot_mae["M5_CleanedHard_CORN"] - pivot_mae["M4_Baseline_CORN"],
    "M6_minus_M4": pivot_mae["M6_CleanedSevere_CORN"] - pivot_mae["M4_Baseline_CORN"],
})
display(diff_df.round(4))

out_path = os.path.join(config.RESULTS_DIR, f"per_class_breakdown__{config.PROXY_NAME}__{config.DATA_VERSION}.csv")
breakdown_df.to_csv(out_path, index=False)
print(f"\n💾 Tersimpan -> {out_path}")

⚡ Memakai true_labels/preds_per_seed yang sudah ada dari Cell 9.
📊 MAE per kelas rating asli, per skenario (rata-rata 3 seed):



scenario,M4_Baseline_CORN,M5_CleanedHard_CORN,M6_CleanedSevere_CORN
rating_asli,,,
1,0.7339,0.6717,0.6255
2,0.5478,0.7754,0.8006
3,0.8747,0.9346,0.9695
4,0.8858,0.8563,0.9388
5,0.5938,0.5081,0.4155



📊 Selisih MAE vs M4 -- POSITIF = cleaning MEMBUAT LEBIH BURUK di kelas itu:



,M5_minus_M4,M6_minus_M4
rating_asli,,
1,-0.0622,-0.1085
2,0.2276,0.2528
3,0.0599,0.0948
4,-0.0296,0.0530
5,-0.0856,-0.1782



💾 Tersimpan -> /content/drive/MyDrive/SKRIPSI_CORN/results/per_class_breakdown__finetuned_corn__v2.csv


In [27]:
# ==========================================================
# CELL 9.6 (BARU): CEK JUMLAH SAMPEL TEST PER KELAS RATING
# ==========================================================
# Memastikan selisih MAE di Cell 9.5 bukan artefak sampel kecil --
# semua kelas idealnya di atas ~250 baris supaya bisa dipercaya.
# ==========================================================
display(breakdown_df[breakdown_df["scenario"] == "M4_Baseline_CORN"][["rating_asli", "n_test"]])

,rating_asli,n_test
0,1,461
1,2,331
2,3,306
3,4,327
4,5,288


In [28]:
# ==========================================================
# CELL 10: RINGKASAN AKHIR -- SIAP DISALIN KE BAB 4
# ==========================================================
import pandas as pd
from src import config

print("=" * 70)
print(f" RINGKASAN LENGKAP -- backbone: {config.PRETRAINED_MODEL_NAME} | data: {config.DATA_VERSION} ")
print("=" * 70)

print("\n[1] TABEL ABLASI PROXY (semua data_version) -- untuk Bab 1:")
if os.path.exists(config.PROXY_QUALITY_LOG_FILE):
    proxy_table = pd.read_csv(config.PROXY_QUALITY_LOG_FILE)
    display(proxy_table)
else:
    print("   ⚠️ Belum ada.")

print(f"\n[2] VALIDASI MANUSIA ({config.PROXY_NAME}, {config.DATA_VERSION}):")
if os.path.exists(config.HUMAN_VALIDATION_RESULT_FILE):
    df_human = pd.read_csv(config.HUMAN_VALIDATION_RESULT_FILE)
    counts = df_human["human_verdict"].value_counts()
    total = len(df_human)
    print(f"   Total: {total} | Noise: {counts.get('noise',0)} ({counts.get('noise',0)/total*100:.1f}%) "
          f"| Not noise: {counts.get('not_noise',0)} ({counts.get('not_noise',0)/total*100:.1f}%) "
          f"| Ambiguous: {counts.get('ambiguous',0)} ({counts.get('ambiguous',0)/total*100:.1f}%)")
else:
    print("   ⚠️ Belum ada -- jalankan Cell 7.5 dulu.")

print(f"\n[3] HASIL 6 SKENARIO (Mean ± Std, 3 seed) -- untuk Bab 4:")
if os.path.exists(config.FINAL_RESULTS_TABLE_FILE):
    display(pd.read_csv(config.FINAL_RESULTS_TABLE_FILE))
else:
    print("   ⚠️ Belum ada -- jalankan Cell 8 dulu.")

print(f"\n[4] UJI SIGNIFIKANSI -- untuk Bab 4:")
if os.path.exists(config.SIGNIFICANCE_TEST_FILE):
    display(pd.read_csv(config.SIGNIFICANCE_TEST_FILE))
else:
    print("   ⚠️ Belum ada -- jalankan Cell 9 dulu.")

print(f"\n[5] EFFECT SIZE + CI 95% -- untuk Bab 4:")
effect_size_file = os.path.join(config.RESULTS_DIR, f"effect_sizes__{config.PROXY_NAME}__{config.DATA_VERSION}.csv")
if os.path.exists(effect_size_file):
    display(pd.read_csv(effect_size_file))
else:
    print("   ⚠️ Belum ada -- jalankan Cell 9 dulu.")

print("\n✅ Semua file hasil tersimpan di:", config.RESULTS_DIR)

 RINGKASAN LENGKAP -- backbone: indobenchmark/indobert-base-p1 | data: v2 

[1] TABEL ABLASI PROXY (semua data_version) -- untuk Bab 1:


,proxy_id,proxy_name,proxy_desc,data_version,accuracy,mae,off_by_one,qwk,pct_flagged_noise
0,0,frozen_cls_lr,CLS embedding beku + LR (P1),v2,0.430282,0.936633,0.749744,0.609230,46.853555
1,1,frozen_meanpool_lr,Mean-pool embedding beku + LR (P2),v2,0.430428,0.928019,0.758651,0.612447,47.233173
2,2,finetuned_ce,"IndoBERT fine-tuned K-Fold, CE loss (P3)",v2,0.445174,0.785078,0.830194,0.669671,52.577019
3,3,finetuned_corn,"IndoBERT fine-tuned K-Fold, CORN loss (P4) -- ...",v2,0.451015,0.771938,0.831070,0.682986,51.700978
4,4,finetuned_corn_fusion,IndoBERT+CORN + fusi sentimen (P5),v2,0.454519,0.767557,0.833552,0.694756,50.620529
5,6,finetuned_corn_indobertweet,"IndoBERTweet fine-tuned K-Fold, CORN loss (P6)",v2,0.459337,0.757921,0.836764,0.695553,47.671193



[2] VALIDASI MANUSIA (finetuned_corn, v2):
   ⚠️ Belum ada -- jalankan Cell 7.5 dulu.

[3] HASIL 6 SKENARIO (Mean ± Std, 3 seed) -- untuk Bab 4:


,Model,MAE (↓),RMSE (↓),Acc (↑),Off-by-1 (↑),QWK (↑)
0,M4_Baseline_CORN,0.7285 ± 0.0140,1.0836 ± 0.0340,0.4485 ± 0.0158,0.8618 ± 0.0184,0.7020 ± 0.0011
1,M2_CleanedHard_CE,0.7429 ± 0.0077,1.1293 ± 0.0074,0.4637 ± 0.0052,0.8432 ± 0.0036,0.6985 ± 0.0074
2,M6_CleanedSevere_CORN,0.7453 ± 0.0060,1.1312 ± 0.0082,0.4639 ± 0.0039,0.8391 ± 0.0040,0.7085 ± 0.0038
3,M5_CleanedHard_CORN,0.7464 ± 0.0051,1.1237 ± 0.0090,0.4598 ± 0.0059,0.8379 ± 0.0029,0.6984 ± 0.0061
4,M3_CleanedSevere_CE,0.7546 ± 0.0193,1.1519 ± 0.0375,0.4639 ± 0.0081,0.8385 ± 0.0153,0.6935 ± 0.0112
5,M1_Baseline_CE,0.7838 ± 0.0153,1.1790 ± 0.0293,0.4516 ± 0.0075,0.8237 ± 0.0122,0.6737 ± 0.0101



[4] UJI SIGNIFIKANSI -- untuk Bab 4:


,hypothesis,model_a,model_b,mean_error_a,mean_error_b,p_value_raw,p_value_corrected,signifikan
0,H1_CORN_vs_CE_raw,M4_Baseline_CORN,M1_Baseline_CE,0.728546,0.783810,0.000001,0.000004,Ya
1,H2_SeverityAware_vs_Base,M6_CleanedSevere_CORN,M4_Baseline_CORN,0.745281,0.728546,0.049302,0.049302,Ya
2,H3_HardPrune_vs_Base,M5_CleanedHard_CORN,M4_Baseline_CORN,0.746449,0.728546,0.021451,0.042903,Ya



[5] EFFECT SIZE + CI 95% -- untuk Bab 4:


,model_a,model_b,mean_diff,ci_95_low,ci_95_high
0,M4_Baseline_CORN,M1_Baseline_CE,-0.055264,-0.076095,-0.035021
1,M6_CleanedSevere_CORN,M4_Baseline_CORN,0.016735,-0.002140,0.035805
2,M5_CleanedHard_CORN,M4_Baseline_CORN,0.017902,-0.001557,0.038334



✅ Semua file hasil tersimpan di: /content/drive/MyDrive/SKRIPSI_CORN/results
